# 11 - Biological interpretation of PCA axes

This notebook tests which biological properties are associated with the 10D PCA coordinates used in the 3D plot and in the GA/KMeans clustering comparison.

Primary biological source: the paper Excel table `mbo006184236st1.xls`, sheet `novel_pago_fixed_july2018`.

Important interpretation rule: PCA axes were computed from SWeeP embeddings, not from biological annotations. Therefore, each PC is reported as statistically associated with biological variables, not as directly measuring them.

In [10]:
# =============================================================================
# CELL 1 - Imports and project root
# =============================================================================

from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pago_pipeline.pc_biology_interpretation import (
    ASSOCIATIONS_FILE_NAME,
    CLUSTER_ENRICHMENT_FILE_NAME,
    INTEGRATED_TABLE_FILE_NAME,
    PLOT_HTML_FILE_NAME,
    SUMMARY_FILE_NAME,
    create_pc_biology_interpretation_snapshot,
)

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Programming\Python\pAgo-project


In [11]:
# =============================================================================
# CELL 2 - Configuration
# =============================================================================

EXCEL_FILE_PATH = Path("data/01-raw/Protein_acession_version/mbo006184236st1.xls")
EXCEL_SHEET_NAME = "novel_pago_fixed_july2018"

PCA_KMEANS_LATEST_DIRECTORY = Path("data/04-analysis/pca_kmeans/latest")
GA_LATEST_DIRECTORY = Path("data/04-analysis/ga_clustering/latest")
QC_LATEST_DIRECTORY = Path("data/03-features/pago_qc/evidence_inventory/latest")
OUTPUT_ROOT_DIRECTORY = Path("data/04-analysis/pc_biology_interpretation")

PERMUTATION_COUNT = 999
RANDOM_STATE = 42
MINIMUM_CATEGORY_COUNT = 5

print(f"Paper Excel source: {PROJECT_ROOT / EXCEL_FILE_PATH}")
print(f"Output root: {PROJECT_ROOT / OUTPUT_ROOT_DIRECTORY}")


Paper Excel source: C:\Programming\Python\pAgo-project\data\01-raw\Protein_acession_version\mbo006184236st1.xls
Output root: C:\Programming\Python\pAgo-project\data\04-analysis\pc_biology_interpretation


## Analysis design

The snapshot integrates PCA coordinates, KMeans labels, GA labels, paper annotations, and QC evidence. The tests include:

- Spearman association with permutation p-values for continuous variables.
- Kruskal-Wallis tests with epsilon squared for categorical variables.
- Length-residualized PC tests to separate length effects from clade/domain effects.
- Chi-square enrichment with Cramer's V for GA/KMeans clusters versus biological categories.
- Benjamini-Hochberg FDR correction across reported tests.

In [12]:
# =============================================================================
# CELL 3 - Build reproducible snapshot
# =============================================================================

result = create_pc_biology_interpretation_snapshot(
    project_root=PROJECT_ROOT,
    excel_file_path=EXCEL_FILE_PATH,
    excel_sheet_name=EXCEL_SHEET_NAME,
    pca_kmeans_latest_directory=PCA_KMEANS_LATEST_DIRECTORY,
    ga_latest_directory=GA_LATEST_DIRECTORY,
    qc_latest_directory=QC_LATEST_DIRECTORY,
    output_root_directory=OUTPUT_ROOT_DIRECTORY,
    permutation_count=PERMUTATION_COUNT,
    random_state=RANDOM_STATE,
    minimum_category_count=MINIMUM_CATEGORY_COUNT,
    update_latest_directory=True,
    verbose=True,
)

print(f"Immutable snapshot: {result.snapshot_directory}")
print(f"Latest snapshot: {result.latest_directory}")


Building integrated PC/biology table...
Computing PC/variable association tests...
Computing AG/KMeans cluster enrichment tests...
Rendering markdown and HTML report...
Snapshot directory: C:\Programming\Python\pAgo-project\data\04-analysis\pc_biology_interpretation\snapshots\2026-06-13T15-54-35Z__q_642d7489fee3
Latest directory: C:\Programming\Python\pAgo-project\data\04-analysis\pc_biology_interpretation\latest
Immutable snapshot: C:\Programming\Python\pAgo-project\data\04-analysis\pc_biology_interpretation\snapshots\2026-06-13T15-54-35Z__q_642d7489fee3
Latest snapshot: C:\Programming\Python\pAgo-project\data\04-analysis\pc_biology_interpretation\latest


In [13]:
# =============================================================================
# CELL 4 - Load generated artifacts
# =============================================================================

latest_directory = result.latest_directory

integrated_table = pd.read_csv(latest_directory / INTEGRATED_TABLE_FILE_NAME, low_memory=False)
associations = pd.read_csv(latest_directory / ASSOCIATIONS_FILE_NAME)
cluster_enrichment = pd.read_csv(latest_directory / CLUSTER_ENRICHMENT_FILE_NAME)

print(f"Integrated rows: {len(integrated_table)}")
print(f"Association tests: {len(associations)}")
print(f"Cluster enrichment tests: {len(cluster_enrichment)}")

display(integrated_table.head())


Integrated rows: 1010
Association tests: 460
Cluster enrichment tests: 24


,sequence_index,record_id,description,sequence_length,protein_uid,gbseq__accession_version,gbseq__comment,gbseq__contig,gbseq__create_date,gbseq__definition,...,qc__length_bin,qc__has_paz_region,qc__has_mid_region,qc__has_piwi_region,qc__has_active_site_annotation,qc__has_cdd_region,qc__primary_label,qc__qc_decision,qc__confidence_score,paper_length_minus_ncbi_sequence_length
0,0,protein_uid=15606619|accession=NP_213999.1|len...,protein_uid=15606619|accession=NP_213999.1|len...,706,15606619,NP_213999.1,REVIEWED REFSEQ: This record has been curated ...,join(WP_010880937.1:1..706),07-SEP-2001,hypothetical protein aq_1447 [Aquifex aeolicus...,...,600_900,True,False,True,False,True,classic_piwi_candidate,include,6,0
1,1,protein_uid=16519675|accession=NP_443795.1|len...,protein_uid=16519675|accession=NP_443795.1|len...,517,16519675,NP_443795.1,PROVISIONAL REFSEQ: This record has not yet be...,join(WP_010875054.1:1..517),22-NOV-1996,hypothetical protein NGR_a00110 (plasmid) [Sin...,...,300_599,False,False,True,True,True,classic_piwi_candidate,review,4,0
2,2,protein_uid=21225132|accession=NP_630911.1|len...,protein_uid=21225132|accession=NP_630911.1|len...,681,21225132,NP_630911.1,REVIEWED REFSEQ: This record has been curated ...,join(WP_011031225.1:1..681),28-MAY-2002,hypothetical protein SCO6840 [Streptomyces coe...,...,600_900,False,False,True,True,True,classic_piwi_candidate,include,7,0
3,3,protein_uid=22298491|accession=NP_681738.1|len...,protein_uid=22298491|accession=NP_681738.1|len...,756,22298491,NP_681738.1,REVIEWED REFSEQ: This record has been curated ...,join(WP_011056792.1:1..756),17-AUG-2002,hypothetical protein tll0948 [Thermosynechococ...,...,600_900,False,False,True,False,True,classic_piwi_candidate,include,6,0
4,4,protein_uid=39996463|accession=NP_952414.1|len...,protein_uid=39996463|accession=NP_952414.1|len...,473,39996463,NP_952414.1,REVIEWED REFSEQ: This record has been curated ...,join(WP_010942012.1:1..473),17-DEC-2003,Piwi domain-containing protein [Geobacter sulf...,...,300_599,False,False,True,True,True,classic_piwi_candidate,review,4,0


In [14]:
# =============================================================================
# CELL 5 - Top raw PC associations
# =============================================================================

top_raw_associations = (
    associations.loc[associations["analysis_scope"].eq("raw_pc")]
    .sort_values(["pc_name", "q_value_bh", "effect_size"], ascending=[True, True, False])
    .groupby("pc_name", as_index=False)
    .head(8)
)

display(
    top_raw_associations[
        [
            "pc_name",
            "variable_name",
            "variable_kind",
            "statistic_name",
            "statistic",
            "effect_size_name",
            "effect_size",
            "q_value_bh",
            "direction",
            "top_group_high",
            "top_group_low",
        ]
    ]
)


,pc_name,variable_name,variable_kind,statistic_name,statistic,effect_size_name,effect_size,q_value_bh,direction,top_group_high,top_group_low
170,pc1,taxonomy__04,categorical,kruskal_h,657.504395,epsilon_squared,0.644595,9.807642e-124,category_shift,Methanomada group,Deinococci
171,pc1,taxonomy__03,categorical,kruskal_h,605.917366,epsilon_squared,0.594076,2.922905e-116,category_shift,Thermotogota,Deinococcota
172,pc1,paper_length_bin,categorical,kruskal_h,470.030974,epsilon_squared,0.464246,1.712746e-99,category_shift,901_1300,300_599
173,pc1,qc__length_bin,categorical,kruskal_h,470.030974,epsilon_squared,0.464246,1.712746e-99,category_shift,901_1300,300_599
174,pc1,paper_mid_5p_type,categorical,kruskal_h,218.784423,epsilon_squared,0.213145,2.393897e-44,category_shift,YK,RK
...,...,...,...,...,...,...,...,...,...,...,...
434,pc9,qc__length_bin,categorical,kruskal_h,75.683628,epsilon_squared,0.072250,8.745761e-16,category_shift,901_1300,lt_300
435,pc9,paper_mid_5p_type,categorical,kruskal_h,79.243559,epsilon_squared,0.074021,3.969262e-15,category_shift,HK,YY
436,pc9,paper_ago_type_family,categorical,kruskal_h,20.706584,epsilon_squared,0.017601,2.731333e-04,category_shift,longB,other_rare
437,pc9,paper_ago_type_raw,categorical,kruskal_h,26.939083,epsilon_squared,0.020876,3.256416e-04,category_shift,longB,longB_trun


In [15]:
# =============================================================================
# CELL 6 - Associations after removing the linear length effect
# =============================================================================

top_length_residual_associations = (
    associations.loc[associations["analysis_scope"].eq("length_residual_pc")]
    .sort_values(["pc_name", "q_value_bh", "effect_size"], ascending=[True, True, False])
    .groupby("pc_name", as_index=False)
    .head(8)
)

display(
    top_length_residual_associations[
        [
            "pc_name",
            "variable_name",
            "effect_size_name",
            "effect_size",
            "q_value_bh",
            "top_group_high",
            "top_group_low",
        ]
    ]
)


,pc_name,variable_name,effect_size_name,effect_size,q_value_bh,top_group_high,top_group_low
0,pc1,taxonomy__04,epsilon_squared,0.489335,3.982820e-92,Thermotogae,Deinococci
1,pc1,taxonomy__03,epsilon_squared,0.438245,4.267769e-84,Thermotogota,Deinococcota
2,pc1,paper_is_truncated,epsilon_squared,0.043873,5.499198e-11,True,False
3,pc1,paper_ago_type_raw,epsilon_squared,0.054281,1.136681e-10,short_trun,short
4,pc1,paper_mid_5p_type,epsilon_squared,0.046085,2.223591e-09,YK,other_rare
...,...,...,...,...,...,...,...
156,pc9,paper_length_bin,epsilon_squared,0.039029,9.658505e-09,901_1300,600_900
157,pc9,qc__length_bin,epsilon_squared,0.039029,9.658505e-09,901_1300,600_900
158,pc9,paper_ago_type_family,epsilon_squared,0.037145,2.381271e-08,short,other_rare
159,pc9,paper_ago_type_raw,epsilon_squared,0.037761,2.057772e-07,short_trun,longB_trun


In [16]:
# =============================================================================
# CELL 7 - Cluster enrichment summary
# =============================================================================

display(
    cluster_enrichment.head(20)[
        [
            "cluster_column",
            "biological_variable",
            "effect_size_name",
            "effect_size",
            "q_value_bh",
        ]
    ]
)


,cluster_column,biological_variable,effect_size_name,effect_size,q_value_bh
0,ga_cluster_label,taxonomy__04,cramers_v,0.468987,0.000000e+00
1,ga_cluster_label,paper_mid_5p_type,cramers_v,0.512809,1.478243e-288
2,ga_cluster_label,taxonomy__03,cramers_v,0.389924,1.671490e-244
3,ga_cluster_label,paper_length_bin,cramers_v,0.583785,3.895726e-195
4,ga_cluster_label,paper_ago_type_family,cramers_v,0.501144,3.030800e-138
5,ga_cluster_label,paper_paz_type,cramers_v,0.591582,6.408273e-135
6,ga_cluster_label,paper_ago_type_raw,cramers_v,0.363685,1.570218e-126
7,ga_cluster_label,qc__qc_decision,cramers_v,0.476380,4.948477e-83
8,ga_cluster_label,paper_domain_architecture,cramers_v,0.469597,2.156620e-80
9,ga_cluster_label,paper_has_piwi_catalytic_tetrad,cramers_v,0.458080,3.114469e-39


In [17]:
# =============================================================================
# CELL 8 - Markdown interpretation summary
# =============================================================================

summary_text = (latest_directory / SUMMARY_FILE_NAME).read_text(encoding="utf-8")
display(Markdown(summary_text))


# PC biological interpretation summary

Interpretation rule: PCs were computed from SWeeP embeddings, not from the biological annotations below. Therefore, a PC should be described as associated with a property, not as directly measuring that property.

Record count: 1010

## PC1

Top continuous associations:
- paper_piwi_start_relative: rho=0.622, q=0.001479, direction=positive
- paper_piwi_length_relative: rho=-0.621, q=0.001479, direction=negative
- paper_length: rho=0.567, q=0.001479, direction=positive

Top categorical associations:
- taxonomy__04: epsilon_squared=0.645, q=9.808e-124, high=Methanomada group, low=Deinococci
- taxonomy__03: epsilon_squared=0.594, q=2.923e-116, high=Thermotogota, low=Deinococcota
- paper_length_bin: epsilon_squared=0.464, q=1.713e-99, high=901_1300, low=300_599

Top categorical associations after length residualization:
- taxonomy__04: epsilon_squared=0.489, q=3.983e-92, high=Thermotogae, low=Deinococci
- taxonomy__03: epsilon_squared=0.438, q=4.268e-84, high=Thermotogota, low=Deinococcota
- paper_is_truncated: epsilon_squared=0.044, q=5.499e-11, high=True, low=False

## PC2

Top continuous associations:
- paper_mid_length_relative: rho=-0.472, q=0.001479, direction=negative
- paper_mid_start_relative: rho=0.462, q=0.001479, direction=positive
- paper_piwi_length: rho=0.343, q=0.001479, direction=positive

Top categorical associations:
- paper_mid_5p_type: epsilon_squared=0.232, q=3.326e-48, high=HK, low=RK
- paper_length_bin: epsilon_squared=0.204, q=5.841e-44, high=901_1300, low=300_599
- qc__length_bin: epsilon_squared=0.204, q=5.841e-44, high=901_1300, low=300_599

Top categorical associations after length residualization:
- paper_mid_5p_type: epsilon_squared=0.220, q=9.136e-46, high=HK, low=YY
- paper_length_bin: epsilon_squared=0.187, q=3.058e-40, high=901_1300, low=600_900
- qc__length_bin: epsilon_squared=0.187, q=3.058e-40, high=901_1300, low=600_900

## PC3

Top continuous associations:
- paper_mid_length: rho=-0.612, q=0.001479, direction=negative
- paper_mid_length_relative: rho=-0.492, q=0.001479, direction=negative
- paper_mid_start_relative: rho=0.464, q=0.001479, direction=positive

Top categorical associations:
- paper_mid_5p_type: epsilon_squared=0.430, q=3.589e-90, high=KR, low=RK
- paper_ago_type_family: epsilon_squared=0.338, q=2.257e-72, high=longB, low=short
- paper_ago_type_raw: epsilon_squared=0.340, q=2.928e-70, high=longB, low=short

Top categorical associations after length residualization:
- paper_mid_5p_type: epsilon_squared=0.327, q=3.362e-68, high=YK, low=RK
- paper_ago_type_family: epsilon_squared=0.271, q=3.415e-58, high=longB, low=short
- paper_ago_type_raw: epsilon_squared=0.277, q=2.924e-57, high=longB, low=short

## PC4

Top continuous associations:
- paper_length: rho=-0.405, q=0.001479, direction=negative
- sequence_length: rho=-0.405, q=0.001479, direction=negative
- paper_piwi_length_relative: rho=0.361, q=0.001479, direction=positive

Top categorical associations:
- paper_mid_5p_type: epsilon_squared=0.232, q=3.325e-48, high=RK, low=HK
- paper_length_bin: epsilon_squared=0.165, q=1.255e-35, high=300_599, low=901_1300
- qc__length_bin: epsilon_squared=0.165, q=1.255e-35, high=300_599, low=901_1300

Top categorical associations after length residualization:
- paper_mid_5p_type: epsilon_squared=0.257, q=1.481e-53, high=RK, low=HK
- taxonomy__04: epsilon_squared=0.153, q=2.023e-25, high=Methanomada group, low=Deinococci
- paper_length_bin: epsilon_squared=0.087, q=5.411e-19, high=600_900, low=901_1300

## PC5

Top continuous associations:
- paper_mid_length: rho=-0.454, q=0.001479, direction=negative
- qc__confidence_score: rho=0.375, q=0.001479, direction=positive
- paper_piwi_length: rho=-0.275, q=0.001479, direction=negative

Top categorical associations:
- paper_mid_5p_type: epsilon_squared=0.314, q=1.255e-65, high=YY, low=RK
- paper_ago_type_family: epsilon_squared=0.270, q=4.787e-58, high=longA, low=short
- paper_length_bin: epsilon_squared=0.270, q=6.408e-58, high=600_900, low=901_1300

Top categorical associations after length residualization:
- paper_mid_5p_type: epsilon_squared=0.357, q=1.4e-74, high=YY, low=RK
- paper_ago_type_family: epsilon_squared=0.317, q=5.109e-68, high=longA, low=short
- paper_ago_type_raw: epsilon_squared=0.316, q=2.362e-65, high=longA, low=short

## PC6

Top continuous associations:
- paper_mid_length_relative: rho=-0.409, q=0.001479, direction=negative
- paper_piwi_length_relative: rho=-0.392, q=0.001479, direction=negative
- paper_mid_start_relative: rho=0.392, q=0.001479, direction=positive

Top categorical associations:
- taxonomy__04: epsilon_squared=0.324, q=7.006e-59, high=Deinococci, low=Gammaproteobacteria
- taxonomy__03: epsilon_squared=0.255, q=5.447e-47, high=Deinococcota, low=Thermotogota
- paper_paz_type: epsilon_squared=0.200, q=7.192e-44, high=normal, low=small

Top categorical associations after length residualization:
- taxonomy__04: epsilon_squared=0.385, q=7.04e-71, high=Deinococci, low=Flavobacteriia
- taxonomy__03: epsilon_squared=0.293, q=1.565e-54, high=Deinococcota, low=Thermotogota
- paper_paz_type: epsilon_squared=0.133, q=1.824e-29, high=normal, low=small

## PC7

Top continuous associations:
- paper_mid_length: rho=0.341, q=0.001479, direction=positive
- paper_mid_length_relative: rho=0.328, q=0.001479, direction=positive
- qc__confidence_score: rho=-0.283, q=0.001479, direction=negative

Top categorical associations:
- taxonomy__04: epsilon_squared=0.269, q=5.419e-48, high=Bacilli, low=Deinococci
- taxonomy__03: epsilon_squared=0.224, q=9.735e-41, high=Bacillota, low=Deinococcota
- paper_ago_type_family: epsilon_squared=0.186, q=5.268e-40, high=short, low=longB

Top categorical associations after length residualization:
- taxonomy__04: epsilon_squared=0.280, q=4.705e-50, high=Bacilli, low=Deinococci
- taxonomy__03: epsilon_squared=0.229, q=1.177e-41, high=Bacillota, low=Deinococcota
- paper_mid_5p_type: epsilon_squared=0.156, q=1.994e-32, high=HK, low=YY

## PC8

Top continuous associations:
- paper_paz_length: rho=0.243, q=0.001479, direction=positive
- paper_paz_length_relative: rho=0.234, q=0.001479, direction=positive
- paper_mid_length: rho=0.151, q=0.001479, direction=positive

Top categorical associations:
- taxonomy__04: epsilon_squared=0.138, q=1.721e-22, high=Actinomycetes, low=Methanomada group
- taxonomy__03: epsilon_squared=0.099, q=2.591e-16, high=Actinomycetota, low=Spirochaetota
- paper_mid_5p_type: epsilon_squared=0.065, q=2.539e-13, high=RK, low=YY

Top categorical associations after length residualization:
- taxonomy__04: epsilon_squared=0.157, q=3.639e-26, high=Actinomycetes, low=Methanomada group
- taxonomy__03: epsilon_squared=0.114, q=2.93e-19, high=Actinomycetota, low=Spirochaetota
- paper_mid_5p_type: epsilon_squared=0.076, q=1.492e-15, high=RK, low=YY

## PC9

Top continuous associations:
- paper_length: rho=0.206, q=0.001479, direction=positive
- sequence_length: rho=0.206, q=0.001479, direction=positive
- paper_piwi_start_relative: rho=0.193, q=0.001479, direction=positive

Top categorical associations:
- taxonomy__04: epsilon_squared=0.211, q=1.136e-36, high=Bacilli, low=Cyanophyceae
- taxonomy__03: epsilon_squared=0.155, q=3.248e-27, high=Deinococcota, low=Cyanobacteriota
- paper_length_bin: epsilon_squared=0.072, q=8.746e-16, high=901_1300, low=lt_300

Top categorical associations after length residualization:
- taxonomy__04: epsilon_squared=0.237, q=1.298e-41, high=Bacilli, low=Cyanophyceae
- taxonomy__03: epsilon_squared=0.174, q=8.468e-31, high=Deinococcota, low=Cyanobacteriota
- paper_mid_5p_type: epsilon_squared=0.088, q=4.816e-18, high=HK, low=YY

## PC10

Top continuous associations:
- paper_paz_start_relative: rho=-0.449, q=0.001479, direction=negative
- paper_paz_length_relative: rho=-0.346, q=0.001479, direction=negative
- paper_piwi_length: rho=0.320, q=0.001479, direction=positive

Top categorical associations:
- taxonomy__04: epsilon_squared=0.263, q=7.873e-47, high=Actinomycetes, low=Cyanophyceae
- paper_ago_type_family: epsilon_squared=0.181, q=5.361e-39, high=longB, low=longA
- paper_ago_type_raw: epsilon_squared=0.182, q=3.169e-37, high=longB_trun, low=longA

Top categorical associations after length residualization:
- taxonomy__04: epsilon_squared=0.262, q=1.34e-46, high=Actinomycetes, low=Cyanophyceae
- paper_ago_type_family: epsilon_squared=0.180, q=8.873e-39, high=longB, low=longA
- paper_ago_type_raw: epsilon_squared=0.181, q=5.524e-37, high=longB_trun, low=longA

## Cluster enrichment
- ga_cluster_label vs taxonomy__04: Cramer's V=0.469, q=0
- ga_cluster_label vs paper_mid_5p_type: Cramer's V=0.513, q=1.478e-288
- ga_cluster_label vs taxonomy__03: Cramer's V=0.390, q=1.671e-244
- ga_cluster_label vs paper_length_bin: Cramer's V=0.584, q=3.896e-195
- ga_cluster_label vs paper_ago_type_family: Cramer's V=0.501, q=3.031e-138
- ga_cluster_label vs paper_paz_type: Cramer's V=0.592, q=6.408e-135
- ga_cluster_label vs paper_ago_type_raw: Cramer's V=0.364, q=1.57e-126
- ga_cluster_label vs qc__qc_decision: Cramer's V=0.476, q=4.948e-83
- ga_cluster_label vs paper_domain_architecture: Cramer's V=0.470, q=2.157e-80
- ga_cluster_label vs paper_has_piwi_catalytic_tetrad: Cramer's V=0.458, q=3.114e-39


In [18]:
# =============================================================================
# CELL 9 - Interactive HTML report
# =============================================================================

html_report_path = latest_directory / PLOT_HTML_FILE_NAME
display(
    HTML(
        f'<p><a href="{html_report_path.as_posix()}" target="_blank">'
        'Open PC biological interpretation HTML report</a></p>'
    )
)

print(html_report_path)


C:\Programming\Python\pAgo-project\data\04-analysis\pc_biology_interpretation\latest\pc_biology_3d.html
